In [1]:
!pip install openai python-dotenv

In [2]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")
print("Chave carregada:", api_key is not None and len(api_key) > 0)

Chave carregada: True


### Carregamento Híbrido de Credenciais (Colab vs. Local)

Para que este código rode localmente sem quebras, tentamos primeiro buscar as chaves usando o `google.colab.userdata`. Se falhar (o que ocorre quando rodamos fora do Colab), carregamos as variáveis de um arquivo `.env` usando a biblioteca `python-dotenv` ou diretamente do sistema operacional (`os.environ`).

In [14]:
import os
from dotenv import load_dotenv

# Tenta carregar o arquivo .env se ele existir (útil localmente)
load_dotenv()

api_key_router = None

try:
    # Se estiver no Colab, tenta buscar dos Secrets internos
    from google.colab import userdata
    api_key_router = userdata.get("OPENROUTER_API_KEY")
except (ImportError, Exception):
    # Se não estiver no Colab ou ocorrer outro erro, busca do ambiente/arquivo .env
    api_key_router = os.getenv("OPENROUTER_API_KEY")

if not api_key_router:
    print("⚠️ Alerta: A chave 'OPENROUTER_API_KEY' não foi encontrada!")
    print("Para rodar localmente, crie um arquivo chamado '.env' na mesma pasta do seu script com o seguinte conteúdo:")
    print("OPENROUTER_API_KEY=sua_chave_aqui")
else:
    print("✅ Chave carregada com sucesso! (Modo Híbrido Ativo)")

✅ Chave carregada com sucesso! (Modo Híbrido Ativo)


In [3]:
import os
import glob
import sqlite3
import zipfile

PASTA_BASE = os.getcwd()   # pasta do notebook (no Colab, /content)
zip_path = os.path.join(PASTA_BASE, "cinerocket-db.zip")
extract_path = os.path.join(PASTA_BASE, "cinerocket_extracted")

if os.path.exists(zip_path):
    print("Extraindo o arquivo ZIP...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)
    db_files = glob.glob(os.path.join(extract_path, "**", "*.db"), recursive=True)
else:
    # Fallback: um .db solto na pasta do notebook
    db_files = glob.glob(os.path.join(PASTA_BASE, "*.db"))

if not db_files:
    raise FileNotFoundError(
        "Banco não encontrado. Coloque 'cinerocket-db.zip' na mesma pasta do notebook."
    )

db_path = db_files[0]
caminho_db = db_path   # usado pela função executar_sql
print(f"Banco encontrado: {db_path}")

# Conexão da exploração/gabarito (o agente abre a própria conexão read-only a cada consulta)
conn = sqlite3.connect(db_path)
print("Tabelas:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")])

Extraindo o arquivo ZIP...
Banco encontrado: /content/cinerocket_extracted/cinerocket (1).db
Tabelas: ['alembic_version', 'dim_companies', 'dim_genres', 'dim_movies', 'dim_people', 'bridge_movie_company', 'bridge_movie_genre', 'bridge_movie_person', 'dim_reviews', 'fact_movies_performance', 'movie_reviews']


In [4]:
conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)

print(conn.execute("SELECT DISTINCT tipo_pessoa FROM dim_people").fetchall())
print(conn.execute("SELECT DISTINCT status_filme FROM dim_movies").fetchall())
print(conn.execute("SELECT nome_genero FROM dim_genres ORDER BY nome_genero").fetchall())
print(conn.execute("SELECT MAX(data_lancamento) FROM dim_movies WHERE data_lancamento <= date('now')").fetchone())
print(conn.execute("SELECT sk_movie_id, titulo FROM dim_movies LIMIT 3").fetchall())

[('Roteirista',), ('Diretor',), ('Ator',)]
[('Lançado',), ('Pós-Produção',), ('Em Produção',), ('Planejado',)]
[('Action',), ('Adventure',), ('Animation',), ('Comedy',), ('Crime',), ('Documentary',), ('Drama',), ('Family',), ('Fantasy',), ('History',), ('Horror',), ('Music',), ('Mystery',), ('Romance',), ('Science Fiction',), ('Thriller',), ('Tv Movie',), ('War',), ('Western',)]
('2026-02-19',)
[('b5c312c4d8cee94972412b9ea785976f9dbb2c26dd11480d9f2b9b4691df1571', 'Rings'), ('d286f479f6764d7a09951df2c51cc95b7615c4f021fedc31773b69fa3e4da5f1', 'Mixtape'), ('54ecb594e9ce04d4b5ba09215e8d130683c7fe6d8ef5f0d469691ae1dfe6df62', 'Grizzly Ii: Revenge')]


In [5]:
import re
import sqlite3

def executar_sql(query: str) -> str:
    """Executa uma query SQL somente-leitura e devolve o resultado como texto."""
    query_normalizada = query.strip().rstrip(";")

    # Guardrail 1: só um único SELECT
    if not re.match(r"(?i)^\s*SELECT\b", query_normalizada):
        return "ERRO: Só são permitidas consultas SELECT."
    if ";" in query_normalizada:
        return "ERRO: Múltiplos comandos SQL não são permitidos."
    palavras_proibidas = r"\b(DROP|DELETE|INSERT|UPDATE|ALTER|ATTACH|PRAGMA|CREATE|REPLACE|VACUUM)\b"
    if re.search(palavras_proibidas, query_normalizada, re.IGNORECASE):
        return "ERRO: Comando não permitido."

    # Guardrail 2: conexão somente leitura, aberta a cada chamada (segura entre threads)
    con = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        cursor = con.cursor()
        cursor.execute(query_normalizada)
        colunas = [d[0] for d in cursor.description]
        linhas = cursor.fetchmany(50)
        return str([dict(zip(colunas, linha)) for linha in linhas])
    except Exception as e:
        return f"ERRO ao executar query: {e}"
    finally:
        con.close()

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "executar_sql",
            "description": "Executa uma consulta SQL de leitura (SELECT) no banco de dados do catálogo de filmes e retorna os resultados.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A consulta SQL SELECT a ser executada."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

In [7]:
from openai import OpenAI
from google.colab import userdata
import json
import os
import time

# Obtendo a chave diretamente do userdata do Colab para evitar falhas de ambiente
api_key_router = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key_router,
)

MODELO = "poolside/laguna-s-2.1:free"
MODELOS_FALLBACK = [
    "poolside/laguna-xs-2.1:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
]

SYSTEM_PROMPT = """Você é um analista de dados do catálogo de filmes da CineData Analytics.
Você responde perguntas em português traduzindo-as em consultas SQL (dialeto SQLite) e
explicando o resultado de forma clara para pessoas não técnicas.

# TABELAS

dim_movies (um registro por filme)
  sk_movie_id (PK), id_filme, titulo, data_lancamento (DATE), ano_lancamento (INT),
  duracao_minutos, idioma_original, status_filme ('Lançado', 'Pós-Produção', 'Em Produção' ou 'Planejado'), sinopse, url_poster, url_backdrop

fact_movies_performance (um registro por filme; FK sk_movie_id -> dim_movies)
  orcamento_usd, receita_usd, lucro_usd, orcamento_brl, receita_brl, lucro_brl,
  popularidade, nota_tmdb, qtd_tmdb, nota_imdb, qtd_imdb

dim_genres: sk_genre_id (PK), nome_genero
dim_people: sk_person_id (PK), nome_pessoa, tipo_pessoa ('Ator', 'Diretor' ou 'Roteirista')
dim_companies: sk_company_id (PK), nome_produtora

bridge_movie_genre:   sk_movie_id, sk_genre_id
bridge_movie_person:  sk_movie_id, sk_person_id
bridge_movie_company: sk_movie_id, sk_company_id

dim_reviews (resumo das avaliações de usuários, um registro por filme)
  sk_review_id, sk_movie_id, qtd_avaliacoes_usuarios, nota_media_usuarios

movie_reviews (avaliações individuais)
  id, sk_movie_review_id, sk_movie_id, name, rating, text, created_at

# RELACIONAMENTOS
- Gênero, pessoa e produtora se ligam ao filme SOMENTE pelas tabelas bridge
  (dim_movies -> bridge_* -> dim_*).
- Métricas financeiras e de engajamento estão em fact_movies_performance
  (join por sk_movie_id).

# REGRAS DE NEGÓCIO
- "Receita", "faturamento" e "bilheteria" são sinônimos: use receita_usd / receita_brl.
  Se o usuário pedir "em R$" ou "em reais", use as colunas *_brl; caso contrário, use *_usd.
- Colunas financeiras e de nota podem ser NULL. Quando a pergunta falar em filmes "com receita
  informada", "com orçamento informado" etc., filtre explicitamente com IS NOT NULL.
  Em médias e somas, ignore NULLs (não os trate como zero).
- Margem de lucro (%) = lucro_usd / orcamento_usd * 100, apenas onde orcamento_usd IS NOT NULL
  e orcamento_usd > 0.
- Divergência entre notas = ABS(nota_tmdb - nota_imdb), apenas onde ambas não são NULL.
- Recortes como "últimos N anos" são relativos à MAIOR data_lancamento da base
  (ignorando datas futuras), e NÃO à data de hoje:
  data_lancamento >= date((SELECT MAX(data_lancamento) FROM dim_movies
                           WHERE data_lancamento <= date('now')), '-N years').
- Os nomes dos gêneros estão em inglês e são exatamente estes: Action, Adventure, Animation,
  Comedy, Crime, Documentary, Drama, Family, Fantasy, History, Horror, Music, Mystery,
  Romance, Science Fiction, Thriller, Tv Movie, War, Western.
  Traduza o que o usuário disser (ex.: "ação" -> 'Action', "terror" -> 'Horror',
  "ficção científica" -> 'Science Fiction') e use o nome exato da lista.
- Contagens de filmes por gênero/pessoa/produtora devem usar COUNT(DISTINCT sk_movie_id).
- Em "mínimo de N filmes", use HAVING COUNT(DISTINCT sk_movie_id) >= N.
- Sempre use LIMIT (padrão 10, ou o número pedido) para não retornar listas enormes.

# COMO RESPONDER
- Use SOMENTE SELECT. Nunca tente alterar o banco.
- Use a ferramenta executar_sql para consultar; não invente números.
- Se a consulta retornar erro, corrija a query e tente de novo (no máximo 2 vezes).
- Se a pergunta não puder ser respondida com estes dados, diga isso claramente.
- Responda em português, de forma direta, citando os valores encontrados.
  Valores em reais no formato R$ 1.234.567,89.

# EXEMPLOS DE CONSULTAS CORRETAS

-- Dupla ator-diretor que mais trabalhou junta
SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor,
       COUNT(DISTINCT bm_a.sk_movie_id) AS qtd_filmes
FROM bridge_movie_person bm_a
JOIN dim_people a ON a.sk_person_id = bm_a.sk_person_id AND a.tipo_pessoa = 'Ator'
JOIN bridge_movie_person bm_d ON bm_d.sk_movie_id = bm_a.sk_movie_id
JOIN dim_people d ON d.sk_person_id = bm_d.sk_person_id AND d.tipo_pessoa = 'Diretor'
GROUP BY a.nome_pessoa, d.nome_pessoa
ORDER BY qtd_filmes DESC
LIMIT 1;

-- Ator com mais participações nos últimos 5 anos
SELECT p.nome_pessoa, COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes
FROM dim_movies m
JOIN bridge_movie_person bp ON bp.sk_movie_id = m.sk_movie_id
JOIN dim_people p ON p.sk_person_id = bp.sk_person_id AND p.tipo_pessoa = 'Ator'
WHERE m.data_lancamento <= date('now')
  AND m.data_lancamento >= date((SELECT MAX(data_lancamento) FROM dim_movies
                                 WHERE data_lancamento <= date('now')), '-5 years')
GROUP BY p.nome_pessoa
ORDER BY qtd_filmes DESC
LIMIT 1;

-- Lucro médio por gênero, apenas filmes com receita informada
SELECT g.nome_genero, AVG(f.lucro_usd) AS lucro_medio_usd
FROM fact_movies_performance f
JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
WHERE f.receita_usd IS NOT NULL AND f.lucro_usd IS NOT NULL
GROUP BY g.nome_genero
ORDER BY lucro_medio_usd DESC;
"""

def chamar_modelo(mensagens, max_tokens=4000, tentativas=3, tool_choice="auto"):
    ultimo_erro = None
    for i in range(tentativas):
        resposta = client.chat.completions.create(
            model=MODELO,
            messages=mensagens,
            tools=tools,
            tool_choice=tool_choice,
            max_tokens=max_tokens,
            extra_body={"models": MODELOS_FALLBACK},
        )
        if resposta and getattr(resposta, "choices", None):
            return resposta

        erro = getattr(resposta, "error", None) or {}
        ultimo_erro = erro
        codigo = erro.get("code") if isinstance(erro, dict) else None

        if codigo in (429, 502, 503, 504) and i < tentativas - 1:
            espera = 2 ** (i + 1)
            print(f"Provedor indisponível (código {codigo}). Tentando de novo em {espera}s...")
            time.sleep(espera)
            continue
        break

    raise RuntimeError(f"Resposta vazia ou inválida da API da OpenRouter: {ultimo_erro}")

def perguntar_ao_agente(pergunta_usuario, historico=None, debug=True, max_rodadas=3):
    mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
    if historico:
        mensagens += historico[-6:]
    mensagens.append({"role": "user", "content": pergunta_usuario})

    for rodada in range(max_rodadas):
        resposta = chamar_modelo(mensagens)
        mensagem = resposta.choices[0].message
        if debug:
            print(f"[rodada {rodada + 1}] modelo={resposta.model} "
                  f"finish={resposta.choices[0].finish_reason} "
                  f"tool_calls={bool(mensagem.tool_calls)}")

        # Sem chamada de ferramenta: se veio texto, é a resposta final
        if not mensagem.tool_calls:
            if mensagem.content and mensagem.content.strip():
                return mensagem.content
            break   # veio vazio: cai na resposta forçada abaixo

        mensagens.append(mensagem)
        for tool_call in mensagem.tool_calls:
            try:
                args = json.loads(tool_call.function.arguments)
                print("SQL gerado:", args["query"])
                resultado_query = executar_sql(args["query"])
            except (json.JSONDecodeError, KeyError) as e:
                resultado_query = f"ERRO: argumentos inválidos na chamada da ferramenta ({e}). Envie apenas o campo 'query' com um SELECT."

            mensagens.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": resultado_query
            })

    # Acabaram as rodadas (ou o modelo veio vazio): força a resposta final, sem novas ferramentas
    mensagens.append({
        "role": "user",
        "content": "Com base nos resultados já obtidos, responda agora em português. Não chame mais ferramentas."
    })
    resposta = chamar_modelo(mensagens, tool_choice="none")
    return resposta.choices[0].message.content

In [8]:
print(executar_sql("SELECT titulo, popularidade FROM dim_movies m JOIN fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id ORDER BY popularidade DESC LIMIT 5"))

[{'titulo': 'Blue Beetle', 'popularidade': 2994.357}, {'titulo': 'Gran Turismo', 'popularidade': 2680.593}, {'titulo': 'La Fellinette', 'popularidade': 2020.0}, {'titulo': 'The Fear Footage 2: Curse Of The Tape', 'popularidade': 2019.0}, {'titulo': 'Wwe Survivor Series 2018', 'popularidade': 2018.0}]


In [9]:
print(perguntar_ao_agente("Quais são os 5 filmes mais populares?"))

[rodada 1] modelo=poolside/laguna-xs-2.1:free finish=tool_calls tool_calls=True
SQL gerado: SELECT m.titulo, f.popularidade
FROM dim_movies m
JOIN fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5;
[rodada 2] modelo=nvidia/nemotron-3-ultra-550b-a55b:free finish=stop tool_calls=False
Os 5 filmes mais populares do catálogo, segundo o índice de popularidade da TMDB:

1. **Blue Beetle** – 2.994,36
2. **Gran Turismo** – 2.680,59
3. **La Fellinette** – 2.020,00
4. **The Fear Footage 2: Curse Of The Tape** – 2.019,00
5. **Wwe Survivor Series 2018** – 2.018,00

A métrica "popularidade" reflete o engajamento e buscas na base TMDB (quanto maior, mais popular).


In [10]:
!pip install gradio

In [11]:
import gradio as gr
import traceback

def texto_do_conteudo(c):
    """O Gradio às vezes entrega o conteúdo como lista de blocos, não como string."""
    if isinstance(c, list):
        return " ".join(
            b.get("text", "") if isinstance(b, dict) else str(b) for b in c
        )
    return str(c)

def normalizar_historico(historico):
    msgs = []
    for item in historico or []:
        if isinstance(item, dict):
            msgs.append({"role": item["role"], "content": texto_do_conteudo(item["content"])})
        else:  # formato antigo: (usuario, assistente)
            usuario, assistente = item
            msgs.append({"role": "user", "content": texto_do_conteudo(usuario)})
            if assistente:
                msgs.append({"role": "assistant", "content": texto_do_conteudo(assistente)})
    return msgs

def responder(mensagem, historico):
    try:
        resposta = perguntar_ao_agente(texto_do_conteudo(mensagem), normalizar_historico(historico))
        if not isinstance(resposta, str) or not resposta.strip():
            return "O modelo não retornou texto desta vez. Tente reformular ou perguntar de novo."
        return resposta
    except Exception as e:
        traceback.print_exc()   # aparece no output da célula do Colab
        return f"Não consegui responder agora ({e}). Tente de novo em instantes."

gr.ChatInterface(
    fn=responder,
    title="CineData Analytics: pergunte sobre o catálogo de filmes",
).launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e08ec9f67ecbe8e055.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e08ec9f67ecbe8e055.gradio.live


In [12]:
import pandas as pd

JOIN_FATO = "FROM fact_movies_performance f JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id"

GABARITO = {
"01 Top 10 filmes por receita em R$": f"""
SELECT m.titulo, f.receita_brl {JOIN_FATO}
WHERE f.receita_brl IS NOT NULL ORDER BY f.receita_brl DESC LIMIT 10""",

"02 Lucro médio por gênero (só com receita informada)": """
SELECT g.nome_genero, ROUND(AVG(f.lucro_usd),2) AS lucro_medio_usd
FROM fact_movies_performance f
JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
WHERE f.receita_usd IS NOT NULL AND f.lucro_usd IS NOT NULL
GROUP BY g.nome_genero ORDER BY lucro_medio_usd DESC""",

"03 Maior margem de lucro (receita e orçamento informados)": f"""
SELECT m.titulo, f.orcamento_usd, f.receita_usd,
       ROUND(f.lucro_usd * 100.0 / f.orcamento_usd, 2) AS margem_pct {JOIN_FATO}
WHERE f.receita_usd IS NOT NULL AND f.orcamento_usd IS NOT NULL AND f.orcamento_usd > 0
ORDER BY margem_pct DESC LIMIT 10""",

"04 Os 5 filmes mais populares": f"""
SELECT m.titulo, f.popularidade {JOIN_FATO}
WHERE f.popularidade IS NOT NULL ORDER BY f.popularidade DESC LIMIT 5""",

"05 Maior divergência nota TMDB x IMDb": f"""
SELECT m.titulo, f.nota_tmdb, f.nota_imdb,
       ROUND(ABS(f.nota_tmdb - f.nota_imdb), 2) AS divergencia {JOIN_FATO}
WHERE f.nota_tmdb IS NOT NULL AND f.nota_imdb IS NOT NULL
ORDER BY divergencia DESC LIMIT 10""",

"06 Nota média IMDb por ano de lançamento": f"""
SELECT m.ano_lancamento, ROUND(AVG(f.nota_imdb), 2) AS nota_media_imdb, COUNT(*) AS qtd_filmes {JOIN_FATO}
WHERE f.nota_imdb IS NOT NULL AND m.ano_lancamento IS NOT NULL
GROUP BY m.ano_lancamento ORDER BY m.ano_lancamento""",

"07 Ator com mais filmes nos últimos 5 anos": """
SELECT p.nome_pessoa, COUNT(DISTINCT m.sk_movie_id) AS qtd_filmes
FROM dim_movies m
JOIN bridge_movie_person bp ON bp.sk_movie_id = m.sk_movie_id
JOIN dim_people p ON p.sk_person_id = bp.sk_person_id AND p.tipo_pessoa = 'Ator'
WHERE m.data_lancamento <= date('now')
  AND m.data_lancamento >= date((SELECT MAX(data_lancamento) FROM dim_movies
                                 WHERE data_lancamento <= date('now')), '-5 years')
GROUP BY p.sk_person_id, p.nome_pessoa ORDER BY qtd_filmes DESC LIMIT 1""",

"08 Diretores com maior nota média IMDb (mín. 5 filmes)": """
SELECT p.nome_pessoa, ROUND(AVG(f.nota_imdb), 2) AS nota_media,
       COUNT(DISTINCT f.sk_movie_id) AS qtd_filmes
FROM dim_people p
JOIN bridge_movie_person bp ON bp.sk_person_id = p.sk_person_id
JOIN fact_movies_performance f ON f.sk_movie_id = bp.sk_movie_id
WHERE p.tipo_pessoa = 'Diretor' AND f.nota_imdb IS NOT NULL
GROUP BY p.sk_person_id, p.nome_pessoa
HAVING COUNT(DISTINCT f.sk_movie_id) >= 5
ORDER BY nota_media DESC LIMIT 10""",

"09 Dupla ator-diretor que mais trabalhou junta": """
SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor,
       COUNT(DISTINCT ba.sk_movie_id) AS qtd_filmes
FROM bridge_movie_person ba
JOIN dim_people a ON a.sk_person_id = ba.sk_person_id AND a.tipo_pessoa = 'Ator'
JOIN bridge_movie_person bd ON bd.sk_movie_id = ba.sk_movie_id
JOIN dim_people d ON d.sk_person_id = bd.sk_person_id AND d.tipo_pessoa = 'Diretor'
GROUP BY a.sk_person_id, d.sk_person_id, a.nome_pessoa, d.nome_pessoa
ORDER BY qtd_filmes DESC LIMIT 1""",

"10 Quantidade de filmes por gênero": """
SELECT g.nome_genero, COUNT(DISTINCT bg.sk_movie_id) AS qtd_filmes
FROM bridge_movie_genre bg JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
GROUP BY g.nome_genero ORDER BY qtd_filmes DESC""",

"11 Produtora com maior lucro total": """
SELECT c.nome_produtora, ROUND(SUM(f.lucro_usd), 2) AS lucro_total_usd
FROM fact_movies_performance f
JOIN bridge_movie_company bc ON bc.sk_movie_id = f.sk_movie_id
JOIN dim_companies c ON c.sk_company_id = bc.sk_company_id
WHERE f.lucro_usd IS NOT NULL
GROUP BY c.sk_company_id, c.nome_produtora ORDER BY lucro_total_usd DESC LIMIT 1""",

"12 Gênero com maior margem de lucro média": """
SELECT g.nome_genero,
       ROUND(AVG(f.lucro_usd * 100.0 / f.orcamento_usd), 2) AS margem_media_pct
FROM fact_movies_performance f
JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
WHERE f.lucro_usd IS NOT NULL AND f.orcamento_usd IS NOT NULL AND f.orcamento_usd > 0
GROUP BY g.nome_genero ORDER BY margem_media_pct DESC LIMIT 1""",

"13 Filmes mais avaliados pelos usuários": """
SELECT m.titulo, r.qtd_avaliacoes_usuarios
FROM dim_reviews r JOIN dim_movies m ON m.sk_movie_id = r.sk_movie_id
ORDER BY r.qtd_avaliacoes_usuarios DESC LIMIT 10""",

"14 Maior divergência nota usuários x IMDb": """
SELECT m.titulo, r.nota_media_usuarios, f.nota_imdb,
       ROUND(ABS(r.nota_media_usuarios - f.nota_imdb), 2) AS divergencia
FROM dim_reviews r
JOIN dim_movies m ON m.sk_movie_id = r.sk_movie_id
JOIN fact_movies_performance f ON f.sk_movie_id = r.sk_movie_id
WHERE r.nota_media_usuarios IS NOT NULL AND f.nota_imdb IS NOT NULL
ORDER BY divergencia DESC LIMIT 10""",
}

for nome, sql in GABARITO.items():
    print(f"\n=== {nome} ===")
    try:
        display(pd.read_sql_query(sql, conn))
    except Exception as e:
        print("ERRO:", e)


=== 01 Top 10 filmes por receita em R$ ===


,titulo,receita_brl
0,Avatar: The Way Of Water,1.239014e+10
1,Avengers: Endgame,1.109472e+10
2,Spider-man: No Way Home,1.097778e+10
3,Avengers: Infinity War,7.190431e+09
4,Top Gun: Maverick,7.160805e+09
5,Barbie,6.856159e+09
6,The Super Mario Bros. Movie,6.838414e+09
7,The Lion King,6.227552e+09
8,Frozen Ii,6.094028e+09
9,Jurassic World: Fallen Kingdom,4.934823e+09



=== 02 Lucro médio por gênero (só com receita informada) ===


,nome_genero,lucro_medio_usd
0,Adventure,1.282366e+08
1,Science Fiction,1.247350e+08
2,Action,8.446762e+07
3,Fantasy,8.374882e+07
4,Family,8.139299e+07
5,Animation,7.280631e+07
6,War,4.731178e+07
7,Comedy,3.490894e+07
8,History,3.408481e+07
9,Music,3.002323e+07



=== 03 Maior margem de lucro (receita e orçamento informados) ===


...

,nome_genero,margem_media_pct
0,Family,40658.45



=== 13 Filmes mais avaliados pelos usuários ===


,titulo,qtd_avaliacoes_usuarios
0,Die Hart 2: Die Harter,13
1,Die Hart 2: Die Harter,12
2,Die Hart: Die Harter,11
3,Die Hart: Die Harter,10
4,Die Hart: Die Harter,10
5,Die Hart: Die Harter,10
6,Die Hart 2: Die Harter,10
7,Die Hart 2: Die Harter,9
8,Die Hart 2: Die Harter,9
9,Die Hart 2: Die Harter,9



=== 14 Maior divergência nota usuários x IMDb ===


,titulo,nota_media_usuarios,nota_imdb,divergencia
0,The Moon Child,0.00,9.8,9.80
1,Milla: The Movie,9.50,0.0,9.50
2,Nathan For You: Finding Frances,0.10,9.5,9.40
3,Opus Cope: An Algorithmic Opera,0.00,9.3,9.30
4,Butterfly,0.40,9.6,9.20
5,Venatio,0.10,9.3,9.20
6,702,0.55,9.7,9.15
7,Mired,0.00,9.1,9.10
8,Ivy,0.30,9.4,9.10
9,The Cost Of Silence,0.10,9.2,9.10
